In [1]:
%cd /app

/app


In [2]:
import argparse
import os
import sys

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

import torch
torch.multiprocessing.set_start_method('spawn')

import jax
from lob.encoding import Vocab, Message_Tokenizer

from lob import inference_no_errcorr as inference
from lob.init_train import init_train_state, load_checkpoint, load_metadata, load_args_from_checkpoint

from lob import inference_no_errcorr as inference
import lob.encoding as encoding
import preproc as preproc

import jax.numpy as jnp
import numpy as np

from pathlib import Path
import os

import pandas as pd

import pandas as pd
import plotly.graph_objs as go
import yaml
import pickle

from filtration_utils import summary_table, build_zero_padded_series, plot_midprice_series_with_insertions, prepare_volatility_filtered_series, plot_midprice_series_with_mean_std
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from typing import Callable, Tuple, Optional, List, Dict

import ipywidgets as widgets
from IPython.display import display, clear_output

2025-09-17 16:43:41.438827: W external/xla/xla/service/gpu/nvptx_compiler.cc:718] The NVIDIA driver's CUDA version is 12.8 which is older than the ptxas CUDA version (12.9.41). Because the driver is older than the ptxas version, XLA is disabling parallel compilation, which may slow down compilation. You should update your NVIDIA driver or use the NVIDIA-provided CUDA forward compatibility packages.
2025-09-17 16:43:43.746577: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [3]:
experiment_name = 'exp_85_20250823_020427_buy_1024'  #       exp_85_20250823_020427_1024
filtration_name = 'samples_after_book_filtration_buy_1024_446'

CONFIG_PATH = f"/app/data_saved/{experiment_name}/used_config.yaml"
sample_day_map = pd.read_csv(f'/app/sample_day_map_1024.csv')
filtration_file = f'/app/data_saved/{experiment_name}/{filtration_name}.csv'


# Load YAML config
with open(CONFIG_PATH, 'r') as f:
    config = yaml.safe_load(f)

# Extract values
num_insertions      = config["num_insertions"]
num_coolings        = config["num_coolings"]
midprice_step_size  = config["midprice_step_size"]
hist_msgs           = config["n_messages"]
n_gen_msgs          = config["n_gen_msgs"]
Direction           = config["DIRECTION_i"]

print(f"Aggressive {'buy' if Direction == 0 else 'sell'}\n")
print(f'num_insertions: {num_insertions}')
print(f'num_coolings: {num_coolings}')
print(f'midprice_step_size: {midprice_step_size}')
print(f'hist_msgs: {hist_msgs}')
print(f'n_gen_msgs: {n_gen_msgs}')

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:10                                                                                   │
│                                                                                                  │
│    7                                                                                             │
│    8                                                                                             │
│    9 # Load YAML config                                                                          │
│ ❱ 10 with open(CONFIG_PATH, 'r') as f:                                                           │
│   11 │   config = yaml.safe_load(f)                                                              │
│   12                                                                                             │
│   13 # Extract values                                                                            │
│                                                                                                  │
│ /opt/conda/envs/myenv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:324 in       │
│ _modified_open                                                                                   │
│                                                                                                  │
│    321 │   │   │   "you can use builtins' open."                                                 │
│    322 │   │   )                                                                                 │
│    323 │                                                                                         │
│ ❱  324 │   return io_open(file, *args, **kwargs)                                                 │
│    325                                                                                           │
│    326 class InteractiveShell(SingletonConfigurable):                                            │
│    327 │   """An enhanced, interactive shell for Python."""                                      │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
FileNotFoundError: [Errno 2] No such file or directory: 
'/app/data_saved/exp_85_20250823_020427_buy_1024/used_config.yaml'

In [ ]:
order_volume = config["order_volume"]
order_volume_ratio = config["order_volume_ratio"]
use_relative_volume = config["use_relative_volume"]

print(f'use_relative_volume: {use_relative_volume}')
print(f'order_volume: {order_volume}')
print(f'order_volume_ratio: {order_volume_ratio}')

use_sample_file = config["use_sample_file"]
sample_file_path = config["sample_file_path"]
start_batch = config["start_batch"]
end_batch = config["end_batch"]

print(f'\nuse_sample_file: {use_sample_file}')
print(f'sample_file_path: {sample_file_path}')
print(f'start_batch: {start_batch}')
print(f'end_batch: {end_batch}')

In [ ]:
hist_steps = hist_msgs // midprice_step_size       # 500
gen_steps = n_gen_msgs // midprice_step_size     # 50
gen_block = gen_steps + 1                        # 51

merged = summary_table(experiment_name)
x, all_series = build_zero_padded_series(hist_msgs, n_gen_msgs, midprice_step_size, merged)

# merged = merged[:10]

print(merged)

In [ ]:
x, all_series

In [ ]:
# import pickle

# save_dir = f"/app/data_saved/{experiment_name}"
# os.makedirs(save_dir, exist_ok=True)

# # Save merged
# with open(os.path.join(save_dir, "merged.pkl"), "wb") as f:
#     pickle.dump(merged, f)

# print(f"Merged data saved to: {save_dir}/merged.pkl")



# ----------------------------------------------------

# OREDER-PLAYER

In [ ]:
import os, glob, re
import numpy as np
import pandas as pd

def build_and_merge(folder, batch_prefix, inp_prefix):
    # STEP 1: load every .npy (shape (batch_size, time, feat)) into a DataFrame
    files   = glob.glob(os.path.join(folder, "*.npy"))
    rx_iter = re.compile(rf"{re.escape(batch_prefix)}_\[(.+)\]_iter_(\d+)\.npy$")
    rx_inp  = re.compile(rf"{re.escape(inp_prefix)}_\[(.+)\]\.npy$")
    rec = []
    for f in files:
        nm = os.path.basename(f)
        m  = rx_iter.match(nm)
        if m:
            rng, itr = m.group(1).replace(" ", ""), int(m.group(2))
        else:
            m2 = rx_inp.match(nm)
            if not m2:
                continue
            rng, itr = m2.group(1).replace(" ", ""), 0

        batch = np.load(f)  # shape (batch_size, time, features)
        print(f"Loaded {nm} with shape {batch.shape}")

        rec.append({"range": rng, "iteration": itr, "batch": batch})
    df = pd.DataFrame(rec).sort_values(["range","iteration"]).reset_index(drop=True)

    # STEP 2: parse the comma‐separated list of IDs into Python lists
    df["ids"] = df["range"].str.split(",").apply(lambda L: [int(x) for x in L])

    # explode each batch into one row per sample, с учётом slicing
    rows = []
    for _, r in df.iterrows():
        for idx, sample_id in enumerate(r["ids"]):
            single = r["batch"][idx]   # shape (time, features)

            # ====== здесь происходит нужный slice ======
            if r["iteration"] > 0:
                n_keep = 51 if r["iteration"] <= num_insertions else 50
                single = single[-n_keep:, :]
            # ============================================

            rows.append({
                "id":        sample_id,
                "iteration": r["iteration"],
                "data":      single
            })

    df_sorted = pd.DataFrame(rows).sort_values(["id","iteration"]).reset_index(drop=True)

    merged = []
    for id_val, grp in df_sorted.groupby("id", sort=True):
        arrs = [row.data for _, row in grp.iterrows()]
        big  = np.concatenate(arrs, axis=0)
        merged.append({"id": id_val, "merged_data": big})
    merged_df = pd.DataFrame(merged).sort_values("id").reset_index(drop=True)

    return df, df_sorted, merged_df

b_folder      = f"/app/data_saved/{experiment_name}/b_seq_gen_doubled"
b_batch_pref  = "b_seq_gen_doubled_batch"
b_inp_pref    = "b_seq_inp"

m_folder      = f"/app/data_saved/{experiment_name}/msgs_decoded_doubled"
m_batch_pref  = "msgs_decoded_doubled_batch"
m_inp_pref    = "m_seq_raw_inp"

# Check if dictionaries already exist
save_folder = f"/app/data_saved/{experiment_name}"
b_dict_path = f"{save_folder}/b_dict.pkl"
m_dict_path = f"{save_folder}/m_dict.pkl"


# if there are dictionaries, load them
if os.path.exists(b_dict_path) and os.path.exists(m_dict_path):
    print(f"Loading existing dictionaries from {save_folder}")
    with open(b_dict_path, "rb") as f:
        b_dict = pickle.load(f)
    with open(m_dict_path, "rb") as f:
        m_dict = pickle.load(f)
# if there are no dictionaries, build and merge data, save them
else:
    print("Building and merging data...")
    _, b_sorted, b_merged = build_and_merge(b_folder, b_batch_pref, b_inp_pref)
    _, m_sorted, m_merged = build_and_merge(m_folder, m_batch_pref, m_inp_pref)

    b_dict = { int(r.id): np.array(r.merged_data) for _, r in b_merged.iterrows() }
    m_dict = { int(r.id): np.array(r.merged_data) for _, r in m_merged.iterrows() }

    save_folder = f"/app/data_saved/{experiment_name}"
    os.makedirs(save_folder, exist_ok=True)

    # Save b_dict
    with open(f"{save_folder}/b_dict.pkl", "wb") as f:
        pickle.dump(b_dict, f)

    # Save m_dict
    with open(f"{save_folder}/m_dict.pkl", "wb") as f:
        pickle.dump(m_dict, f)

    print(f"Saved b_dict, m_dict, b_merged, and m_merged to {save_folder}")

for d in (b_dict, m_dict):
    for key, arr in d.items():
        zero = np.zeros((1, arr.shape[1]), dtype=arr.dtype)
        d[key] = np.vstack([zero, arr])

In [ ]:
# Filter dictionaries and merged data based on filtration file
# filtration_file = f"/app/samples_after_book_filtration.csv"

if os.path.exists(filtration_file):
    print(f"Loading filtration samples from {filtration_file}")
    # Read the filtration file
    with open(filtration_file, 'r') as f:
        lines = f.readlines()
    
    # Skip the header line and get sample IDs
    filtered_sample_ids = set()
    for line in lines[1:]:  # Skip first line which is the header
        line = line.strip()
        if line and line.isdigit():
            filtered_sample_ids.add(int(line))
    
    print(f"Found {len(filtered_sample_ids)} samples in filtration file")
    
    # Filter b_dict
    original_b_count = len(b_dict)
    b_dict = {k: v for k, v in b_dict.items() if k in filtered_sample_ids}
    print(f"Filtered b_dict: {original_b_count} -> {len(b_dict)} samples")
    
    # Filter m_dict
    original_m_count = len(m_dict)
    m_dict = {k: v for k, v in m_dict.items() if k in filtered_sample_ids}
    print(f"Filtered m_dict: {original_m_count} -> {len(m_dict)} samples")
    
    # Filter merged (midprices)
    merged = summary_table(experiment_name)
    original_merged_count = len(merged)
    merged = merged[merged['id'].isin(filtered_sample_ids)]
    print(f"Filtered merged (midprices): {original_merged_count} -> {len(merged)} samples")
    
else:
    print(f"Filtration file {filtration_file} not found. Using all samples.")
    # Load merged (midprices) without filtering
    merged = summary_table(experiment_name)


# 1. Midprice change

In [ ]:
import yaml
import numpy as np
import plotly.graph_objects as go

# Build zero-padded series from merged data
# The function returns only 2 values, not 4
result = build_zero_padded_series(
    hist_msgs, n_gen_msgs, midprice_step_size, merged
)

# Unpack the 2 values returned by the function
x, all_series = result

# Calculate hist_steps and gen_block manually
hist_steps = hist_msgs // midprice_step_size
gen_steps = n_gen_msgs // midprice_step_size
gen_block = gen_steps + 1

fig = go.Figure()

# === Mean & Std ===
mean_series = all_series.mean(axis=0)
std_series  = all_series.std(axis=0)

# ±1 std band
fig.add_trace(go.Scatter(
    x=np.concatenate([x, x[::-1]]),
    y=np.concatenate([mean_series + std_series, (mean_series - std_series)[::-1]]),
    fill='toself',
    fillcolor='rgba(255,0,0,0.05)',
    line=dict(color='rgba(0,0,0,0)'),
    hoverinfo='skip',
    showlegend=False
))

# Mean line
fig.add_trace(go.Scatter(
    x=x, y=mean_series, mode='lines',
    name="GenAI Mean",
    line=dict(color='red', width=3)
))

# Add reference lines
fig.add_hline(y=0, line=dict(color='gray', dash='dash'), name="Zero line")

fig.update_layout(
    title="Midprice Mean ±1 Std (GenAI)",
    xaxis_title="Steps (sampled midprice points)",
    yaxis_title="Price – first price",
    template="plotly_white",
    hovermode="x unified",
    height=800,
    width=800,
    legend=dict(x=1.01, y=0.99),
)
fig.show()


# 2. Market Impact graph

In [ ]:
def calculate_impact(messages, valid_insertions, reference_price):
    """
    Calculate market impact for each insertion.
    
    Parameters
    ----------
    messages : np.array
        Message array with columns [EVENT_TYPE, DIRECTION, PRICE, REL, SIZE, ...]
    valid_insertions : list
        List of insertion indices
    reference_price : float
        Reference price at first insertion
        
    Returns
    -------
    impact : np.array
        Absolute impact for each insertion
    vwap_series : np.array
        VWAP series for each insertion
    Q_cum : np.array
        Cumulative quantity for each insertion
    log_imp : np.array
        Log of impact values for plotting
    """
    PRICE_COL = 3
    SIZE_COL = 5
    insert_sizes = messages[valid_insertions, SIZE_COL].astype(float)
    insert_prices = messages[valid_insertions, PRICE_COL].astype(float)
    Q_cum = np.cumsum(insert_sizes)
    notional = np.cumsum(insert_sizes * insert_prices)

    vwap_series = notional / np.maximum(Q_cum, 1e-12)
    # vwap_series = insert_prices
    
    impact = np.abs(vwap_series - reference_price) / reference_price
    
    # Calculate log impact for plotting (y-axis)
    eps = 1e-12
    log_imp = np.log(np.maximum(impact, eps))
    
    return impact, vwap_series, Q_cum, log_imp


def calculate_market_volume(messages, hist_steps, valid_insertions, execution_sum):
    """
    Calculate market execution volume V_exp for each insertion and return x-axis values for plotting.
    
    Parameters
    ----------
    messages : np.array
        Message array with columns [EVENT_TYPE, DIRECTION, PRICE, REL, SIZE, ...]
    hist_steps : int
        Starting index for market volume calculation
    valid_insertions : list
        List of insertion indices
    execution_sum : float
        Total execution sum for the day
        
    Returns
    -------
    V_exp : np.array
        Market volume from hist_steps to (idx-1) for each insertion
    log_qv : np.array
        Log of Q/V_exp ratio for plotting (x-axis)
    """
    EVENT_TYPE_COL = 1
    SIZE_COL = 5
    
    evt_types = messages[:, EVENT_TYPE_COL].astype(int)
    exec_sizes = np.where(evt_types == 4, messages[:, SIZE_COL].astype(float), 0.0)
    cum_exec_vol = np.cumsum(exec_sizes)
    
    V_exp = np.array([float(cum_exec_vol[idx-1] - cum_exec_vol[hist_steps-1] if (idx-1) >= hist_steps else 0.0)
                      for idx in valid_insertions])

    # Use execution_sum instead of fixed value
    V_exp = np.full_like(V_exp, execution_sum)
    
    # Calculate cumulative quantity for Q/V_exp ratio
    insert_sizes = messages[valid_insertions, SIZE_COL].astype(float)
    Q_cum = np.cumsum(insert_sizes)
    
    # Calculate log(Q/V_exp) for plotting (x-axis)
    eps = 1e-12
    rel_size = Q_cum / np.maximum(V_exp, eps)
    log_qv = np.log(np.maximum(rel_size, eps))

    return V_exp, log_qv

In [ ]:
# Different methods of impact regression
import numpy as np

# ---------- small utils ----------
def _as_float(a):
    return np.asarray(a, dtype=float)

def _mask_xy(x, y):
    x = _as_float(x); y = _as_float(y)
    m = np.isfinite(x) & np.isfinite(y) & (x != 0.0)
    return x[m], y[m], m

def _beta_wls_fixed_internal(x, y_adj, w):
    # Weighted regression of y_adj on x with intercept fixed at 0
    x = _as_float(x); y_adj = _as_float(y_adj); w = _as_float(w)
    m = np.isfinite(x) & np.isfinite(y_adj) & np.isfinite(w) & (x != 0) & (w > 0)
    if m.sum() < 2: return np.nan
    xw = x[m] * np.sqrt(w[m]); yw = y_adj[m] * np.sqrt(w[m])
    denom = np.dot(xw, xw)
    if denom <= 0: return np.nan
    return float(np.dot(xw, yw) / denom)

# ---------- estimators (fixed intercept y = alpha + beta * x) ----------
def _beta_ols_fixed(x, y, alpha):
    x, y_adj, _ = _mask_xy(x, _as_float(y) - _as_float(alpha))
    if x.size < 2: return np.nan
    denom = np.dot(x, x)
    if denom <= 0: return np.nan
    return float(np.dot(x, y_adj) / denom)

def _beta_huber_fixed(x, y, alpha, c=1.345, max_iter=50, tol=1e-8):
    # Huber M via IRLS (defaults chosen for ~95% Gaussian efficiency)
    x, y_adj, _ = _mask_xy(x, _as_float(y) - _as_float(alpha))
    if x.size < 2: return np.nan
    beta = _beta_ols_fixed(x, y, alpha)
    if not np.isfinite(beta): beta = 0.0
    for _ in range(max_iter):
        r = y_adj - beta * x
        med = np.median(r)
        sigma = 1.4826 * np.median(np.abs(r - med)) or (np.std(r) + 1e-12)
        u = r / (sigma + 1e-12)
        w = np.ones_like(u)
        big = np.abs(u) > c
        w[big] = (c / (np.abs(u[big]) + 1e-12))
        beta_new = _beta_wls_fixed_internal(x, y_adj, w)
        if not np.isfinite(beta_new): break
        if abs(beta_new - beta) <= tol * (1.0 + abs(beta)):
            beta = beta_new; break
        beta = beta_new
    return float(beta)

def _beta_lad_fixed(x, y, alpha, iters=100, eps=1e-8):
    # LAD (L1) via IRLS: w_i = 1/max(|r_i|, eps)
    x, y_adj, _ = _mask_xy(x, _as_float(y) - _as_float(alpha))
    if x.size < 2: return np.nan
    beta = _beta_ols_fixed(x, y, alpha)
    if not np.isfinite(beta): beta = 0.0
    for _ in range(iters):
        r = y_adj - beta * x
        w = 1.0 / np.maximum(np.abs(r), eps)
        beta_new = _beta_wls_fixed_internal(x, y_adj, w)
        if not np.isfinite(beta_new): break
        if abs(beta_new - beta) <= 1e-7 * (1.0 + abs(beta)):
            beta = beta_new; break
        beta = beta_new
    return float(beta)

def _beta_ratio_median(x, y, alpha, x_floor=1e-6):
    # Median of ratios with |x| floor (avoid blow-ups near 0)
    x = _as_float(x); y_adj = _as_float(y) - _as_float(alpha)
    m = np.isfinite(x) & np.isfinite(y_adj) & (np.abs(x) >= x_floor)
    r = y_adj[m] / x[m]
    if r.size == 0: return np.nan
    return float(np.median(np.sort(r)))

def _beta_ratio_trim(x, y, alpha, x_floor=1e-6, trim=0.10):
    # Trimmed-mean of ratios (default 10% each tail) with |x| floor
    x = _as_float(x); y_adj = _as_float(y) - _as_float(alpha)
    m = np.isfinite(x) & np.isfinite(y_adj) & (np.abs(x) >= x_floor)
    r = np.sort(y_adj[m] / x[m])
    if r.size == 0: return np.nan
    k = int(trim * r.size)
    r = r[k: r.size - k] if r.size - 2*k > 0 else r
    return float(np.mean(r))

def _beta_deming_fixed(x, y, alpha, lambda_yx=1.0):
    # Orthogonal regression with fixed intercept (through origin on y_adj)
    x, y_adj, _ = _mask_xy(x, _as_float(y) - _as_float(alpha))
    if x.size < 2: return np.nan
    s_xx = np.dot(x, x) / x.size
    s_yy = np.dot(y_adj, y_adj) / x.size
    s_xy = np.dot(x, y_adj) / x.size
    if s_xy == 0.0: return np.nan
    A = s_yy - lambda_yx * s_xx
    B = 2.0 * s_xy
    disc = A*A + (B*B) * lambda_yx
    beta = (A + np.sqrt(disc)) / B
    return float(beta)

# ---------- single public entry ----------
def beta_fit(x, y, alpha, method="ols"):
    """
    Estimate beta in y = alpha + beta * x with a fixed intercept.

    Parameters
    ----------
    x, y : array-like
    alpha : float or array-like (broadcastable)
    method : {'ols','huber','lad','ratio-median','ratio-trim','deming'}

    Returns
    -------
    beta : float
    """
    m = method.lower()
    if m == "ols":
        return _beta_ols_fixed(x, y, alpha)
    elif m == "huber":
        return _beta_huber_fixed(x, y, alpha)           # c=1.345, 50 iters, tol=1e-8
    elif m == "lad":
        return _beta_lad_fixed(x, y, alpha)             # 100 iters, eps=1e-8
    elif m == "ratio-median":
        return _beta_ratio_median(x, y, alpha)          # x_floor=1e-6
    elif m == "ratio-trim":
        return _beta_ratio_trim(x, y, alpha)            # trim=10%, x_floor=1e-6
    elif m == "deming":
        return _beta_deming_fixed(x, y, alpha)          # lambda_yx=1.0
    else:
        raise ValueError(f"Unknown method '{method}'. Use one of: "
                         "ols, huber, lad, ratio-median, ratio-trim, deming.")

In [ ]:
def market_impact_dashboard_from_raw(
    b_seq_inp,            # dict[id] -> np.array(T, ...), where col 0 = Δmid per step (ticks)
    msg_seq_raw,          # dict[id] -> np.array(T, num_fields)
    all_series,           # unused for mid now
    x,                    # time axis (len T) - unused here
    hist_steps=550,
    gen_block=50,
    num_insertions=20,
    *,
    beta_theory=0.5,
    samples_used=None,
    special_first=(79, 15),
    show_fit=True,
    tick_size=100,        # convert ticks -> dollars before calling helper funcs
):
    """
    Same UI/figure as before, but x/y are computed via your helper functions:
      - impact/log_imp from calculate_impact(...)
      - V_exp/log_qv from calculate_market_volume(...), now fed with day-level execution_sum from sample_day_map
    α is fixed to ln(eta_day), where eta_day = ln(H/L)/0.8325546 using H,L from the same sample_day_map.
    β is the fixed-intercept slope: mean((y - α) / x).
    """

    # ------------- normalize inputs -------------
    if isinstance(b_seq_inp, pd.DataFrame):
        b_dict_local = {int(r.id): np.array(r.merged_data) for _, r in b_seq_inp.iterrows()}
    else:
        b_dict_local = b_seq_inp
    if isinstance(msg_seq_raw, pd.DataFrame):
        m_dict_local = {int(r.id): np.array(r.merged_data) for _, r in msg_seq_raw.iterrows()}
    else:
        m_dict_local = msg_seq_raw

    EVENT_TYPE_COL = 1
    PRICE_COL      = 3
    SIZE_COL       = 5

    eps = 1e-12
    tol = 1e-12

    # -------------------- compute x_df, y_df, coeffs_df via helpers --------------------
    def compute_tables():
        sample_ids = sorted(set(b_dict_local.keys()) & set(m_dict_local.keys()))
        col_names = [f"ins_{i}" for i in range(1, num_insertions + 1)]
        x_df = pd.DataFrame(index=sample_ids, columns=col_names, dtype=object)
        y_df = pd.DataFrame(index=sample_ids, columns=col_names, dtype=object)
        coeff_rows = []

        for sid in sample_ids:
            messages_ticks = m_dict_local[sid]
            book = b_dict_local[sid]
            T = len(messages_ticks)

            # insertion schedule
            insertion_positions = hist_steps + np.arange(1, num_insertions + 1) * gen_block
            valid_insertions = [pos for pos in insertion_positions if pos < T]
            if not valid_insertions:
                coeff_rows.append({"sample_id": sid, "alpha_hat": np.nan, "beta_hat": np.nan,
                                   "n_used": 0, "n_total": 0})
                continue

            # Reference price at first insertion (ticks -> $)
            ref_idx = valid_insertions[0]
            reference_price = float(messages_ticks[ref_idx, PRICE_COL]) / tick_size

            # -------- NEW: get day info from sample_day_map --------
            # Look up this sample_id in sample_day_map
            try:
                day_row = sample_day_map[sample_day_map['sample_id'] == sid]
                if not day_row.empty:
                    H_ticks = float(day_row.iloc[0]['highest_price'])
                    L_ticks = float(day_row.iloc[0]['lowest_price'])
                    execution_sum = float(day_row.iloc[0]['execution_sum'])
                else:
                    # Fallback if not found: infer H/L from pre-gen window and use window exec sum
                    H_ticks = float(np.max(messages_ticks[:hist_steps, PRICE_COL])) if hist_steps <= T else float(np.max(messages_ticks[:, PRICE_COL]))
                    L_ticks = float(np.min(messages_ticks[:hist_steps, PRICE_COL])) if hist_steps <= T else float(np.min(messages_ticks[:, PRICE_COL]))
                    exec_mask = (messages_ticks[:, EVENT_TYPE_COL].astype(int) == 4)
                    execution_sum = float(np.sum(messages_ticks[exec_mask, SIZE_COL].astype(float)))
            except Exception as _e:
                # Fallback if not found: infer H/L from pre-gen window and use window exec sum
                H_ticks = float(np.max(messages_ticks[:hist_steps, PRICE_COL])) if hist_steps <= T else float(np.max(messages_ticks[:, PRICE_COL]))
                L_ticks = float(np.min(messages_ticks[:hist_steps, PRICE_COL])) if hist_steps <= T else float(np.min(messages_ticks[:, PRICE_COL]))
                exec_mask = (messages_ticks[:, EVENT_TYPE_COL].astype(int) == 4)
                execution_sum = float(np.sum(messages_ticks[exec_mask, SIZE_COL].astype(float)))

            # Convert to dollars for Parkinson eta
            H = float(H_ticks) / tick_size
            L = float(L_ticks) / tick_size
            if np.isfinite(H) and np.isfinite(L) and H > L and L > 0:
                eta_day = np.log(H / L) / 0.8325546
                alpha_fixed = float(np.log(max(eta_day, eps)))   # α = ln(η)
            else:
                eta_day = eps
                alpha_fixed = float(np.log(eta_day))

            # Convert messages to dollars for the helper functions
            messages_dollars = messages_ticks.astype(float).copy()
            messages_dollars[:, PRICE_COL] /= tick_size

            # --- use your helper functions (unchanged graphs) ---
            impact, vwap_series, Q_cum, log_imp = calculate_impact(
                messages_dollars, valid_insertions, reference_price
            )
            # -------- NEW: pass execution_sum from sample_day_map into V_exp calc --------
            V_exp, log_qv = calculate_market_volume(
                messages_dollars, hist_steps, valid_insertions, execution_sum
            )

            # fill tables per insertion (same)
            mask_zero = impact <= tol
            mask_pos  = ~mask_zero
            for j, _idx in enumerate(valid_insertions):
                col = f"ins_{j+1}"
                if mask_zero[j] or not np.isfinite(log_qv[j]) or not np.isfinite(log_imp[j]):
                    x_df.loc[sid, col] = "ZERO"
                    y_df.loc[sid, col] = "ZERO"
                else:
                    x_df.loc[sid, col] = float(log_qv[j])
                    y_df.loc[sid, col] = float(log_imp[j])

            # per-sample β with fixed intercept (same)
            used_x = log_qv[mask_pos]
            used_y = log_imp[mask_pos]
            n_used = int(used_x.size)
            n_total = int(len(valid_insertions))
            if n_used >= 2 and np.all(np.isfinite(used_x)) and np.all(np.isfinite(used_y)):
                valid_mask = (used_x != 0) & np.isfinite(used_x) & np.isfinite(used_y)
                if np.sum(valid_mask) >= 2:
                    # beta_hat = float(np.mean((used_y[valid_mask] - alpha_fixed) / used_x[valid_mask]))
                    beta_hat = beta_fit(used_x[valid_mask], used_y[valid_mask], alpha_fixed, method=est_method)                    
                else:
                    beta_hat = np.nan
            else:
                beta_hat = np.nan

            coeff_rows.append({
                "sample_id": sid,
                "alpha_hat": alpha_fixed,   # fixed ln(η_day) from sample_day_map
                "beta_hat": beta_hat,
                "n_used": n_used,
                "n_total": n_total,
            })

        coeffs_df = pd.DataFrame.from_records(coeff_rows).set_index("sample_id").sort_index()
        return x_df, y_df, coeffs_df

    x_df, y_df, coeffs_df = compute_tables()

    # -------------------- tidy points (unchanged) --------------------
    all_ids = list(x_df.index)
    ordered_ids = [sid for sid in special_first if sid in all_ids]
    ordered_ids += [sid for sid in sorted(all_ids) if sid not in ordered_ids]
    if samples_used is not None:
        ordered_ids = ordered_ids[:samples_used]

    rows = []
    for sid in ordered_ids:
        for j, col in enumerate(x_df.columns, start=1):
            xv = x_df.loc[sid, col]
            yv = y_df.loc[sid, col]
            if isinstance(xv, (int, float, np.floating)) and isinstance(yv, (int, float, np.floating)):
                if np.isfinite(xv) and np.isfinite(yv):
                    rows.append({"sample_id": sid, "insertion": j, "x": float(xv), "y": float(yv)})
    points_df = pd.DataFrame(rows)
    if points_df.empty:
        print("No numeric points to plot.")
        return None, pd.DataFrame(), pd.DataFrame(), {}

    data_max_ins = int(points_df["insertion"].max())
    max_insertions = int(min(num_insertions, data_max_ins))
    a_values = np.arange(1, max_insertions + 1)

    # -------------------- histogram data (unchanged) --------------------
    betas_clean = coeffs_df["beta_hat"].replace([np.inf, -np.inf], np.nan).dropna().astype(float)
    beta_mean = float(betas_clean.mean()) if not betas_clean.empty else np.nan

    # -------------------- figure (unchanged) --------------------
    fig = go.FigureWidget(make_subplots(
        rows=2, cols=2,
        specs=[[{"type": "xy"}, {"type": "xy"}],
               [None,          {"type": "xy"}]],
        column_widths=[0.68, 0.32],
        row_heights=[0.55, 0.45],
        horizontal_spacing=0.07,
        vertical_spacing=0.12,
        subplot_titles=("Scatter & Global Fit", "β(a) evolution", "Distribution of β̂ across samples")
    ))

    LEGEND_MAX = 15
    palette = ["#1f77b4", "#d62728", "#2ca02c", "#9467bd", "#8c564b",
               "#e377c2", "#7f7f7f", "#bcbd22", "#17becf", "#ff7f0e"]
    GREY = "rgba(0,0,0,0.25)"

    trace_meta = []
    for i, sid in enumerate(ordered_ids):
        sub = points_df[points_df["sample_id"] == sid].sort_values("insertion")
        x_vals = sub["x"].to_numpy()
        y_vals = sub["y"].to_numpy()
        ins = sub["insertion"].to_numpy()
        base_color = palette[i % len(palette)] if i != 1 else "#d62728"
        tr = go.Scatter(
            x=x_vals, y=y_vals, mode="markers",
            name=f"sample {sid} ({len(sub)}/{len(sub)} pts)",
            legendgroup=str(sid), showlegend=(i < LEGEND_MAX),
            marker=dict(size=7, color=base_color),
            hovertemplate=(
                "sample=%{customdata[0]}<br>"
                "ins=%{customdata[1]}<br>"
                "log(Q/V)=%{x:.4f}<br>"
                "log(Impact)=%{y:.4f}<extra></extra>"
            ),
            customdata=np.stack([sub["sample_id"].to_numpy(), ins], axis=1),
        )
        fig.add_trace(tr, row=1, col=1)
        trace_meta.append({"sid": sid, "ins": ins, "x": x_vals, "y": y_vals,
                           "base_color": base_color, "total": len(sub)})

    # global-fit trace (same)
    fit_trace_index = len(fig.data)
    fig.add_trace(
        go.Scatter(x=[], y=[], mode="lines", name="Global fit",
                   line=dict(dash="dash", width=2)),
        row=1, col=1
    )

    # ----- fitting helpers (unchanged, uses α_global = mean ln(η)) -----
    alpha_global = float(coeffs_df["alpha_hat"].mean(skipna=True)) if not coeffs_df.empty else 0.0

    def fit_for_mask(mask):
        X = points_df.loc[mask, "x"].to_numpy()
        Y = points_df.loc[mask, "y"].to_numpy()
        if len(Y) < 2:
            return np.nan, np.nan, np.nan, 0, np.array([]), np.array([])
        valid_mask = np.isfinite(X) & np.isfinite(Y) & (X != 0)
        if np.sum(valid_mask) < 2:
            return alpha_global, np.nan, np.nan, len(Y), np.array([]), np.array([])
        x_valid = X[valid_mask]; y_valid = Y[valid_mask]


        # ================================ #

        
        # beta = float(np.mean((y_valid - alpha_global) / x_valid))
        beta = beta_fit(x_valid, y_valid, alpha_global, method=est_method)
        

        # ================================ #
        
        y_pred = alpha_global + beta * X
        ss_tot = float(((Y - Y.mean()) ** 2).sum())
        ss_res = float(((Y - y_pred) ** 2).sum())
        r2 = 1.0 - (ss_res / ss_tot) if ss_tot > 0 else np.nan
        x_min, x_max = X.min(), X.max()
        x_line = np.linspace(x_min, x_max, 200)
        y_line = alpha_global + beta * x_line
        return alpha_global, beta, r2, len(Y), x_line, y_line

    # precompute β(a) & fit lines (unchanged)
    a_values = np.arange(1, max_insertions + 1)
    betas_evo = np.full_like(a_values, np.nan, dtype=float)
    fit_lines = {}
    for idx, a in enumerate(a_values):
        mask = points_df["insertion"] >= a
        alpha, beta, r2, n, x_line, y_line = fit_for_mask(mask)
        betas_evo[idx] = beta
        fit_lines[a] = (x_line, y_line, alpha, beta, r2, n)

    # β(a) and indicator (unchanged)
    beta_line_idx = len(fig.data)
    fig.add_trace(
        go.Scatter(x=a_values, y=betas_evo, mode="lines+markers", name="β(a)"),
        row=1, col=2
    )
    beta_vline_idx = len(fig.data)
    y_min = float(np.nanmin(betas_evo)) if np.isfinite(betas_evo).any() else 0.0
    y_max = float(np.nanmax(betas_evo)) if np.isfinite(betas_evo).any() else 1.0
    fig.add_trace(
        go.Scatter(x=[a_values[0], a_values[0]], y=[y_min, y_max],
                   mode="lines", line=dict(color="green", dash="dot"), name="current a"),
        row=1, col=2
    )

    # histogram β̂ (unchanged)
    if not betas_clean.empty:
        fig.add_trace(go.Histogram(x=betas_clean.values, nbinsx=30, name="β̂"), row=2, col=2)
        fig.add_trace(go.Scatter(x=[beta_mean, beta_mean], y=[0, max(1, len(betas_clean))],
                                 mode="lines", name=f"Mean β̂ = {beta_mean:.4f}",
                                 line=dict(dash="dash")), row=2, col=2)
        fig.add_trace(go.Scatter(x=[beta_theory, beta_theory], y=[0, max(1, len(betas_clean))],
                                 mode="lines", name=f"β = {beta_theory:.2f} (theoretical)",
                                 line=dict(dash="dot")), row=2, col=2)

    # axes & layout (unchanged)
    fig.update_xaxes(title_text="log(Q / V_exp)", row=1, col=1)
    fig.update_yaxes(title_text="log(Impact)", row=1, col=1)
    fig.update_xaxes(title_text="a (insertion threshold)", row=1, col=2)
    fig.update_yaxes(title_text="β (slope)", row=1, col=2)
    fig.update_xaxes(title_text="β̂", row=2, col=2)
    fig.update_yaxes(title_text="Frequency", row=2, col=2)

    fig.update_layout(template="plotly_white", width=1500, height=800,
                      margin=dict(t=70, r=50, b=60, l=60),
                      legend=dict(orientation="v"))

    # fit box (unchanged)
    def set_fit_annotation(text_html: str):
        fig.layout.annotations = tuple(
            a for a in (fig.layout.annotations or [])
            if getattr(a, "name", "") != "fit_box"
        )
        fig.add_annotation(
            x=0.02, y=0.98, xref="paper", yref="paper",
            text=text_html, showarrow=False, align="left",
            bordercolor="lightgray", borderwidth=1, borderpad=8,
            bgcolor="rgba(245,245,245,1)", name="fit_box"
        )

    def update_left_fit(a_val: int):
        x_line, y_line, alpha, beta, r2, n = fit_lines.get(a_val, ([], [], np.nan, np.nan, np.nan, 0))
        fig.data[fit_trace_index].x = x_line
        fig.data[fit_trace_index].y = y_line
        if show_fit and n > 0 and np.isfinite(beta):
            fit_html = (
                "<b>Market Impact Fit (Fixed Intercept)</b><br>"
                f"<b>a:</b> {a_val}<br>"
                f"<b>Model:</b> log(Impact) = <b>{alpha:.6f}</b> + <b>{beta:.6f}</b> · log(Q/V)<br>"
                f"<b>α (fixed):</b> {alpha:.6f} (mean ln(η) across samples)<br>"
                f"<b>β:</b> {beta:.6f}<br>"
                f"<b>R²:</b> {r2:.4f}<br>"
                f"<b>Used points:</b> {n}"
            )
        else:
            fit_html = f"<b>Market Impact Fit (Fixed Intercept)</b><br><b>a:</b> {a_val}<br>No active points."
        set_fit_annotation(fit_html)

    def recolor_and_refit(a_val: int):
        for t_idx, meta in enumerate(trace_meta):
            active_mask = meta["ins"] >= a_val
            colors = [meta["base_color"] if ok else GREY for ok in active_mask]
            fig.data[t_idx].marker.color = colors
            active_count = int(np.count_nonzero(active_mask))
            fig.data[t_idx].name = f"sample {meta['sid']} ({meta['total']}/{active_count} pts)"
        if show_fit:
            update_left_fit(a_val)

        y_min_local = float(np.nanmin(betas_evo)) if np.isfinite(betas_evo).any() else 0.0
        y_max_local = float(np.nanmax(betas_evo)) if np.isfinite(betas_evo).any() else 1.0
        fig.data[beta_vline_idx].x = [a_val, a_val]
        fig.data[beta_vline_idx].y = [y_min_local, y_max_local]

    # controls (unchanged)
    a_slider = widgets.IntSlider(value=1, min=1, max=max_insertions, step=1, description="a")
    prev_btn = widgets.Button(description="Prev", layout=widgets.Layout(width="80px"))
    next_btn = widgets.Button(description="Next", layout=widgets.Layout(width="80px"))

    def on_prev(_):
        if a_slider.value > a_slider.min:
            a_slider.value -= 1

    def on_next(_):
        if a_slider.value < a_slider.max:
            a_slider.value += 1

    def on_a_change(change):
        if change["name"] == "value":
            recolor_and_refit(change["new"])

    prev_btn.on_click(on_prev)
    next_btn.on_click(on_next)
    a_slider.observe(on_a_change, names="value")

    # initial render
    recolor_and_refit(a_slider.value)
    display(widgets.HBox([prev_btn, next_btn, a_slider]), fig)

    controls = {"a_slider": a_slider, "prev_btn": prev_btn, "next_btn": next_btn}
    return fig, points_df, coeffs_df, controls

In [ ]:
est_method="lad"

# Call the market impact dashboard function
fig, points_df, coeffs_df, controls = market_impact_dashboard_from_raw(
    b_seq_inp=b_dict,
    msg_seq_raw=m_dict,
    all_series=all_series,
    x=x,
    hist_steps=hist_steps,
    gen_block=gen_block,
    num_insertions=num_insertions,
    beta_theory=0.5,
    samples_used=None,
    special_first=(79, 15),
    show_fit=True
)

In [ ]:
# Create beta(a) evolution plot with different regression estimations using Plotly
import plotly.graph_objects as go
import numpy as np

# Define all available estimation methods from beta_fit function
methods = ["ols", "lad", "huber", "ratio-median", "ratio-trim", "deming"]
method_labels = {
    "ols": "OLS", 
    "lad": "LAD", 
    "huber": "Huber",
    "ratio-median": "Ratio Median",
    "ratio-trim": "Ratio Trim",
    "deming": "Deming"
}
colors = ["#1f77b4", "#d62728", "#2ca02c", "#9467bd", "#8c564b", "#e377c2"]

# Create Plotly figure
fig = go.Figure()

# Calculate max_insertions from points_df
max_insertions = points_df["insertion"].max() if not points_df.empty else 10

# Calculate beta evolution for each method
a_values = np.arange(1, max_insertions + 1)

for method_idx, method in enumerate(methods):
    betas_evo = np.full_like(a_values, np.nan, dtype=float)
    
    # Calculate alpha_global (same for all methods)
    alpha_global = float(coeffs_df["alpha_hat"].mean(skipna=True)) if not coeffs_df.empty else 0.0
    
    for idx, a in enumerate(a_values):
        # Filter points for current threshold
        mask = points_df["insertion"] >= a
        X = points_df.loc[mask, "x"].to_numpy()
        Y = points_df.loc[mask, "y"].to_numpy()
        
        if len(Y) >= 2:
            valid_mask = np.isfinite(X) & np.isfinite(Y) & (X != 0)
            if np.sum(valid_mask) >= 2:
                x_valid = X[valid_mask]
                y_valid = Y[valid_mask]
                try:
                    beta = beta_fit(x_valid, y_valid, alpha_global, method=method)
                    betas_evo[idx] = beta
                except Exception:
                    betas_evo[idx] = np.nan
    
    # Add trace for this method
    fig.add_trace(go.Scatter(
        x=a_values, 
        y=betas_evo, 
        mode='lines+markers',
        name=f'{method_labels[method]} regression',
        line=dict(color=colors[method_idx], width=2),
        marker=dict(size=6, color=colors[method_idx])
    ))

# Add theoretical beta line
fig.add_trace(go.Scatter(
    x=[a_values[0], a_values[-1]], 
    y=[0.5, 0.5],
    mode='lines',
    name='Theoretical β = 0.5',
    line=dict(color='black', dash='dash', width=2)
))

# Customize layout
fig.update_layout(
    title=dict(
        text='Beta Evolution with Different Regression Methods',
        font=dict(size=16, family="Arial Black")
    ),
    xaxis=dict(
        title='a (insertion threshold)',
        title_font=dict(size=14),
        tickfont=dict(size=12)
    ),
    yaxis=dict(
        title='β (slope)',
        title_font=dict(size=14),
        tickfont=dict(size=12),
        range=[0.0, 1.0]
    ),
    template='plotly_white',
    width=800,
    height=800,
    legend=dict(
        font=dict(size=12),
        orientation="v",
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=0.01
    ),
    showlegend=True
)

# Add grid
fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='rgba(128,128,128,0.3)')
fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='rgba(128,128,128,0.3)')

fig.show()


In [ ]:
# Create an interactive plotly widget to switch between insertion points with all estimation methods shown simultaneously
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display

# First, prepare data for all insertions
# Group by sample_id and create insertion numbers based on order
points_df_with_insertion = points_df.copy()
points_df_with_insertion['insertion_number'] = points_df_with_insertion.groupby('sample_id').cumcount() + 1
max_insertions = points_df_with_insertion['insertion_number'].max()

# Calculate alpha_global
alpha_global = float(coeffs_df["alpha_hat"].mean(skipna=True)) if not coeffs_df.empty else 0.0

# Available estimation methods
estimation_methods = ['ols', 'huber', 'lad', 'ratio-median', 'ratio-trim', 'deming']
method_colors = {
    'ols': 'red',
    'huber': 'blue', 
    'lad': 'orange',
    'ratio-median': 'purple',
    'ratio-trim': 'brown',
    'deming': 'pink'
}

# Prepare data for each insertion with all methods
insertion_data_dict = {}
for insertion_num in range(1, max_insertions + 1):
    insertion_data = points_df_with_insertion[points_df_with_insertion['insertion_number'] == insertion_num]
    
    if len(insertion_data) > 0:
        X = insertion_data["x"].to_numpy()
        Y = insertion_data["y"].to_numpy()
        
        # Filter out invalid points
        valid_mask = np.isfinite(X) & np.isfinite(Y) & (X != 0)
        x_valid = X[valid_mask]
        y_valid = Y[valid_mask]
        
        # Calculate fitted beta for each method
        beta_fitted_dict = {}
        for method in estimation_methods:
            if len(x_valid) >= 2:
                beta_fitted_dict[method] = beta_fit(x_valid, y_valid, alpha_global, method=method)
            else:
                beta_fitted_dict[method] = np.nan
        
        insertion_data_dict[insertion_num] = {
            'x_all': X,
            'y_all': Y,
            'x_valid': x_valid,
            'y_valid': y_valid,
            'beta_fitted_dict': beta_fitted_dict,
            'n_points': len(X),
            'n_valid': len(x_valid)
        }

# Create FigureWidget for interactive updates
fig = go.FigureWidget()

# Create widgets
insertion_dropdown = widgets.Dropdown(
    options=[(f'Insertion {i}', i) for i in range(1, max_insertions + 1)],
    value=1,
    description='Insertion:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='200px')
)

# Arrow control buttons
prev_button = widgets.Button(
    description='◀ Previous',
    button_style='info',
    layout=widgets.Layout(width='100px')
)

next_button = widgets.Button(
    description='Next ▶',
    button_style='info',
    layout=widgets.Layout(width='100px')
)

# Create output widget for statistics
stats_output = widgets.Output()

def update_plot(insertion_num):
    if insertion_num in insertion_data_dict:
        data = insertion_data_dict[insertion_num]
        
        # Clear existing traces
        with fig.batch_update():
            fig.data = []
            
            # Add scatter plot
            fig.add_scatter(
                x=data['x_all'],
                y=data['y_all'],
                mode='markers',
                name=f'Insertion {insertion_num} points (n={data["n_points"]})',
                marker=dict(size=8, opacity=0.7, color='black'),
                hovertemplate='x=%{x:.6f}<br>y=%{y:.6f}<extra></extra>'
            )
            
            # Add fitted lines for all methods
            if len(data['x_valid']) >= 2 and len(data['x_all']) > 0:
                x_range = np.linspace(data['x_all'].min(), data['x_all'].max(), 100)
                
                for method in estimation_methods:
                    beta_fitted = data['beta_fitted_dict'][method]
                    if np.isfinite(beta_fitted):
                        y_fitted = alpha_global + beta_fitted * x_range
                        fig.add_scatter(
                            x=x_range,
                            y=y_fitted,
                            mode='lines',
                            name=f'{method.upper()}: β = {beta_fitted:.6f}',
                            line=dict(color=method_colors[method], dash='dash', width=2)
                        )
            
            # Add theoretical line
            beta_theory = 0.5
            if len(data['x_all']) > 0:
                x_range = np.linspace(data['x_all'].min(), data['x_all'].max(), 100)
                y_theory = alpha_global + beta_theory * x_range
                fig.add_scatter(
                    x=x_range,
                    y=y_theory,
                    mode='lines',
                    name=f'Theoretical: β = {beta_theory:.6f}',
                    line=dict(color='green', dash='dot', width=3)
                )
            
            # Update layout
            fig.update_layout(
                title=f'Market Impact: Insertion {insertion_num} (All Estimation Methods)',
                xaxis_title='log(Q / V_exp)',
                yaxis_title='log(Impact)',
                template='plotly_white',
                width=1200,
                height=700,
                legend=dict(
                    font=dict(size=11),
                    orientation="v",
                    yanchor="top",
                    y=0.99,
                    xanchor="left",
                    x=0.01
                ),
                showlegend=True
            )
            
            # Add grid
            fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='rgba(128,128,128,0.3)')
            fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='rgba(128,128,128,0.3)')
        
        # Update statistics output
        with stats_output:
            stats_output.clear_output()
            print(f"Insertion {insertion_num} Statistics:")
            print(f"Alpha (fixed intercept): {alpha_global:.6f}")
            print(f"Theoretical Beta: {beta_theory:.6f}")
            print(f"Total points: {data['n_points']}")
            print(f"Valid points: {data['n_valid']}")
            print()
            print("Fitted Beta values by method:")
            for method in estimation_methods:
                beta_fitted = data['beta_fitted_dict'][method]
                print(f"  {method.upper():12}: β = {beta_fitted:.6f}")

# Connect widgets to update function
def on_dropdown_change(change):
    update_plot(insertion_dropdown.value)

def on_prev_click(b):
    current_val = insertion_dropdown.value
    if current_val > 1:
        insertion_dropdown.value = current_val - 1

def on_next_click(b):
    current_val = insertion_dropdown.value
    if current_val < max_insertions:
        insertion_dropdown.value = current_val + 1

insertion_dropdown.observe(on_dropdown_change, names='value')
prev_button.on_click(on_prev_click)
next_button.on_click(on_next_click)

# Initialize with first insertion
update_plot(1)

# Create control panel layout
controls_row = widgets.HBox([prev_button, insertion_dropdown, next_button])
controls = widgets.VBox([controls_row])

# Display widgets and plot
display(widgets.VBox([controls, stats_output, fig]))


In [ ]:
# Calculate mean and std for each insertion across all samples
insertion_stats = []

# First, let's check what columns are available in points_df
print("Available columns in points_df:", points_df.columns.tolist())

# Check if we have insertion_number column, if not, we'll need to create it or use a different approach
if 'insertion_number' not in points_df.columns:
    print("Warning: 'insertion_number' column not found in points_df")
    print("Using sample-based analysis instead...")
    
    # Group by sample_id and create insertion numbers based on order
    points_df_with_insertion = points_df.copy()
    points_df_with_insertion['insertion_number'] = points_df_with_insertion.groupby('sample_id').cumcount() + 1
    max_insertions = points_df_with_insertion['insertion_number'].max()
else:
    points_df_with_insertion = points_df
    max_insertions = points_df['insertion_number'].max()

for insertion_num in range(1, max_insertions + 1):
    # Get data for this insertion number
    insertion_data = points_df_with_insertion[points_df_with_insertion['insertion_number'] == insertion_num]
    
    if len(insertion_data) > 0:
        x_values = insertion_data['x'].to_numpy()
        y_values = insertion_data['y'].to_numpy()
        
        # Filter out invalid points
        valid_mask = np.isfinite(x_values) & np.isfinite(y_values) & (x_values != 0)
        x_valid = x_values[valid_mask]
        y_valid = y_values[valid_mask]
        
        if len(x_valid) > 0:
            x_mean = np.mean(x_valid)
            x_std = np.std(x_valid)
            y_mean = np.mean(y_valid)
            y_std = np.std(y_valid)
            
            insertion_stats.append({
                'insertion_number': insertion_num,
                'x_mean': x_mean,
                'x_std': x_std,
                'y_mean': y_mean,
                'y_std': y_std,
                'n_points': len(x_valid)
            })

# Convert to DataFrame for easier handling
stats_df = pd.DataFrame(insertion_stats)

if len(stats_df) > 0:
    # Create the plotly figure
    fig = go.Figure()

    # Add scatter plot with error bars
    fig.add_trace(go.Scatter(
        x=stats_df['x_mean'],
        y=stats_df['y_mean'],
        error_x=dict(type='data', array=stats_df['x_std'], visible=True),
        error_y=dict(type='data', array=stats_df['y_std'], visible=True),
        mode='markers+text',
        text=[str(int(row['insertion_number'])) for _, row in stats_df.iterrows()],
        textposition='top right',
        textfont=dict(size=10),
        marker=dict(size=8, opacity=0.7),
        name=f'Mean impact by insertion (n={len(stats_df)})',
        hovertemplate='Insertion: %{text}<br>' +
                      'x_mean: %{x:.6f}<br>' +
                      'y_mean: %{y:.6f}<br>' +
                      '<extra></extra>'
    ))

    # Add fitted line if we have enough points
    if len(stats_df) >= 2:
        x_mean_valid = stats_df['x_mean'].to_numpy()
        y_mean_valid = stats_df['y_mean'].to_numpy()
        
        # Fit line to mean values
        beta_insertion = beta_fit(x_mean_valid, y_mean_valid, alpha_global, method=est_method)
        
        if np.isfinite(beta_insertion):
            x_range = np.linspace(x_mean_valid.min(), x_mean_valid.max(), 100)
            y_fitted = alpha_global + beta_insertion * x_range
            fig.add_trace(go.Scatter(
                x=x_range,
                y=y_fitted,
                mode='lines',
                line=dict(color='red', dash='dash', width=2),
                name=f'Fitted: β = {beta_insertion:.6f}',
                hovertemplate='Fitted line<br>' +
                              'x: %{x:.6f}<br>' +
                              'y: %{y:.6f}<br>' +
                              '<extra></extra>'
            ))

    # Add theoretical line
    beta_theory = 0.5
    x_range_theory = np.linspace(stats_df['x_mean'].min(), stats_df['x_mean'].max(), 100)
    y_theory = alpha_global + beta_theory * x_range_theory
    fig.add_trace(go.Scatter(
        x=x_range_theory,
        y=y_theory,
        mode='lines',
        line=dict(color='green', dash='dot', width=2),
        name=f'Theoretical: β = {beta_theory:.6f}',
        hovertemplate='Theoretical line<br>' +
                      'x: %{x:.6f}<br>' +
                      'y: %{y:.6f}<br>' +
                      '<extra></extra>'
    ))

    # Update layout
    fig.update_layout(
        title='Market Impact: Mean by Insertion Number',
        xaxis_title='log(Q / V_exp) - Mean',
        yaxis_title='log(Impact) - Mean',
        width=1000,
        height=600,
        showlegend=True,
        hovermode='closest'
    )

    # Add grid
    fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='rgba(128,128,128,0.3)')
    fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='rgba(128,128,128,0.3)')

    # Display the plot
    fig.show()

    # Print summary statistics
    print(f"Number of insertions analyzed: {len(stats_df)}")
    print(f"Alpha (fixed intercept): {alpha_global:.6f}")
    if len(stats_df) >= 2 and 'beta_insertion' in locals() and np.isfinite(beta_insertion):
        print(f"Fitted Beta (insertion means): {beta_insertion:.6f}")
    print(f"Theoretical Beta: {beta_theory:.6f}")
    print(f"Maximum insertion number: {max_insertions}")
else:
    print("No valid insertion data found for analysis")


In [ ]:
# Calculate mean and std for each insertion across all samples
insertion_stats = []

# First, let's check what columns are available in points_df
print("Available columns in points_df:", points_df.columns.tolist())

# Check if we have insertion_number column, if not, we'll need to create it or use a different approach
if 'insertion_number' not in points_df.columns:
    print("Warning: 'insertion_number' column not found in points_df")
    print("Using sample-based analysis instead...")
    
    # Group by sample_id and create insertion numbers based on order
    points_df_with_insertion = points_df.copy()
    points_df_with_insertion['insertion_number'] = points_df_with_insertion.groupby('sample_id').cumcount() + 1
    max_insertions = points_df_with_insertion['insertion_number'].max()
else:
    points_df_with_insertion = points_df
    max_insertions = points_df['insertion_number'].max()

for insertion_num in range(1, max_insertions + 1):
    # Get data for this insertion number
    insertion_data = points_df_with_insertion[points_df_with_insertion['insertion_number'] == insertion_num]
    
    if len(insertion_data) > 0:
        x_values = insertion_data['x'].to_numpy()
        y_values = insertion_data['y'].to_numpy()
        
        # Filter out invalid points
        valid_mask = np.isfinite(x_values) & np.isfinite(y_values) & (x_values != 0)
        x_valid = x_values[valid_mask]
        y_valid = y_values[valid_mask]
        
        if len(x_valid) > 0:
            x_mean = np.mean(x_valid)
            x_std = np.std(x_valid)
            y_mean = np.mean(y_valid)
            y_std = np.std(y_valid)
            
            insertion_stats.append({
                'insertion_number': insertion_num,
                'x_mean': x_mean,
                'x_std': x_std,
                'y_mean': y_mean,
                'y_std': y_std,
                'n_points': len(x_valid)
            })

# Convert to DataFrame for easier handling
stats_df = pd.DataFrame(insertion_stats)

if len(stats_df) > 0:
    # Create the plotly figure
    fig = go.Figure()

    # Add scatter plot with error bars (only y-axis std)
    fig.add_trace(go.Scatter(
        x=stats_df['x_mean'],
        y=stats_df['y_mean'],
        error_y=dict(type='data', array=stats_df['y_std'], visible=True),
        mode='markers+text',
        text=[str(int(row['insertion_number'])) for _, row in stats_df.iterrows()],
        textposition='top right',
        textfont=dict(size=10),
        marker=dict(size=8, opacity=0.7),
        name=f'Mean impact by insertion (n={len(stats_df)})',
        hovertemplate='Insertion: %{text}<br>' +
                      'x_mean: %{x:.6f}<br>' +
                      'y_mean: %{y:.6f}<br>' +
                      '<extra></extra>'
    ))

    # Add fitted line if we have enough points
    if len(stats_df) >= 2:
        x_mean_valid = stats_df['x_mean'].to_numpy()
        y_mean_valid = stats_df['y_mean'].to_numpy()
        
        # Fit line to mean values
        beta_insertion = beta_fit(x_mean_valid, y_mean_valid, alpha_global, method=est_method)
        
        if np.isfinite(beta_insertion):
            x_range = np.linspace(x_mean_valid.min(), x_mean_valid.max(), 100)
            y_fitted = alpha_global + beta_insertion * x_range
            fig.add_trace(go.Scatter(
                x=x_range,
                y=y_fitted,
                mode='lines',
                line=dict(color='red', dash='dash', width=2),
                name=f'Fitted: β = {beta_insertion:.6f}',
                hovertemplate='Fitted line<br>' +
                              'x: %{x:.6f}<br>' +
                              'y: %{y:.6f}<br>' +
                              '<extra></extra>'
            ))

    # Add theoretical line
    beta_theory = 0.5
    x_range_theory = np.linspace(stats_df['x_mean'].min(), stats_df['x_mean'].max(), 100)
    y_theory = alpha_global + beta_theory * x_range_theory
    fig.add_trace(go.Scatter(
        x=x_range_theory,
        y=y_theory,
        mode='lines',
        line=dict(color='green', dash='dot', width=2),
        name=f'Theoretical: β = {beta_theory:.6f}',
        hovertemplate='Theoretical line<br>' +
                      'x: %{x:.6f}<br>' +
                      'y: %{y:.6f}<br>' +
                      '<extra></extra>'
    ))

    # Update layout
    fig.update_layout(
        title='Market Impact: Mean by Insertion Number',
        xaxis_title='log(Q / V_exp) - Mean',
        yaxis_title='log(Impact) - Mean',
        width=1000,
        height=600,
        showlegend=True,
        hovermode='closest'
    )

    # Add grid
    fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='rgba(128,128,128,0.3)')
    fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='rgba(128,128,128,0.3)')

    # Display the plot
    fig.show()

    # Print summary statistics
    print(f"Number of insertions analyzed: {len(stats_df)}")
    print(f"Alpha (fixed intercept): {alpha_global:.6f}")
    if len(stats_df) >= 2 and 'beta_insertion' in locals() and np.isfinite(beta_insertion):
        print(f"Fitted Beta (insertion means): {beta_insertion:.6f}")
    print(f"Theoretical Beta: {beta_theory:.6f}")
    print(f"Maximum insertion number: {max_insertions}")
else:
    print("No valid insertion data found for analysis")


In [ ]:
# Calculate mean and std for each x-axis bin across all samples
import numpy as np

# First, let's check what columns are available in points_df
print("Available columns in points_df:", points_df.columns.tolist())

# Get all valid x and y values
x_values = points_df['x'].to_numpy()
y_values = points_df['y'].to_numpy()

# Filter out invalid points
valid_mask = np.isfinite(x_values) & np.isfinite(y_values) & (x_values != 0)
x_valid = x_values[valid_mask]
y_valid = y_values[valid_mask]

if len(x_valid) > 0:
    # Create bins for x-axis
    n_bins = 5000  # Number of bins
    x_min, x_max = x_valid.min(), x_valid.max()
    bin_edges = np.linspace(x_min, x_max, n_bins + 1)
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    
    # Calculate statistics for each bin
    bin_stats = []
    
    for i in range(n_bins):
        # Find points in this bin
        bin_mask = (x_valid >= bin_edges[i]) & (x_valid < bin_edges[i + 1])
        if i == n_bins - 1:  # Include the last edge in the final bin
            bin_mask = (x_valid >= bin_edges[i]) & (x_valid <= bin_edges[i + 1])
        
        x_bin = x_valid[bin_mask]
        y_bin = y_valid[bin_mask]
        
        if len(x_bin) > 0:
            x_mean = np.mean(x_bin)
            x_std = np.std(x_bin)
            y_mean = np.mean(y_bin)
            y_std = np.std(y_bin)
            
            bin_stats.append({
                'bin_number': i + 1,
                'bin_center': bin_centers[i],
                'x_mean': x_mean,
                'x_std': x_std,
                'y_mean': y_mean,
                'y_std': y_std,
                'n_points': len(x_bin),
                'x_min': x_bin.min(),
                'x_max': x_bin.max()
            })

    # Convert to DataFrame for easier handling
    stats_df = pd.DataFrame(bin_stats)

    if len(stats_df) > 0:
        # Create the plotly figure
        fig = go.Figure()

        # Add scatter plot with error bars (only y-axis std)
        fig.add_trace(go.Scatter(
            x=stats_df['x_mean'],
            y=stats_df['y_mean'],
            error_y=dict(type='data', array=stats_df['y_std'], visible=True, color='blue'),
            mode='markers',
            marker=dict(size=5, color='black', opacity=1.0),
            name=f'Mean impact by x-bin (n={len(stats_df)})',
            hovertemplate='Bin: %{customdata}<br>' +
                          'x_mean: %{x:.6f}<br>' +
                          'y_mean: %{y:.6f}<br>' +
                          'n_points: %{customdata}<br>' +
                          '<extra></extra>',
            customdata=stats_df['bin_number']
        ))

        # Add fitted line if we have enough points
        if len(stats_df) >= 2:
            x_mean_valid = stats_df['x_mean'].to_numpy()
            y_mean_valid = stats_df['y_mean'].to_numpy()
            
            # Fit line to mean values
            beta_bins = beta_fit(x_mean_valid, y_mean_valid, alpha_global, method=est_method)
            
            if np.isfinite(beta_bins):
                x_range = np.linspace(x_mean_valid.min(), x_mean_valid.max(), 100)
                y_fitted = alpha_global + beta_bins * x_range
                fig.add_trace(go.Scatter(
                    x=x_range,
                    y=y_fitted,
                    mode='lines',
                    line=dict(color='red', dash='dash', width=2),
                    name=f'Fitted: β = {beta_bins:.6f}',
                    hovertemplate='Fitted line<br>' +
                                  'x: %{x:.6f}<br>' +
                                  'y: %{y:.6f}<br>' +
                                  '<extra></extra>'
                ))

        # Add theoretical line
        beta_theory = 0.5
        x_range_theory = np.linspace(stats_df['x_mean'].min(), stats_df['x_mean'].max(), 100)
        y_theory = alpha_global + beta_theory * x_range_theory
        fig.add_trace(go.Scatter(
            x=x_range_theory,
            y=y_theory,
            mode='lines',
            line=dict(color='green', dash='dot', width=2),
            name=f'Theoretical: β = {beta_theory:.6f}',
            hovertemplate='Theoretical line<br>' +
                          'x: %{x:.6f}<br>' +
                          'y: %{y:.6f}<br>' +
                          '<extra></extra>'
        ))

        # Update layout
        fig.update_layout(
            title='Market Impact: Mean by X-Axis Bins',
            xaxis_title='log(Q / V_exp) - Mean',
            yaxis_title='log(Impact) - Mean',
            width=1000,
            height=600,
            showlegend=True,
            hovermode='closest'
        )

        # Add grid
        fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='rgba(128,128,128,0.3)')
        fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='rgba(128,128,128,0.3)')

        # Display the plot
        fig.show()

        # Print summary statistics
        print(f"Number of x-bins analyzed: {len(stats_df)}")
        print(f"Number of bins: {n_bins}")
        print(f"Total points used: {len(x_valid)}")
        print(f"Alpha (fixed intercept): {alpha_global:.6f}")
        if len(stats_df) >= 2 and 'beta_bins' in locals() and np.isfinite(beta_bins):
            print(f"Fitted Beta (x-bin means): {beta_bins:.6f}")
        print(f"Theoretical Beta: {beta_theory:.6f}")
        print(f"X-axis range: [{x_min:.6f}, {x_max:.6f}]")
        
        # Print bin details
        print("\nBin details:")
        for _, row in stats_df.iterrows():
            print(f"Bin {int(row['bin_number'])}: x=[{row['x_min']:.6f}, {row['x_max']:.6f}], "
                  f"x_mean={row['x_mean']:.6f}, y_mean={row['y_mean']:.6f}, n={int(row['n_points'])}")
    else:
        print("No valid bin data found for analysis")
else:
    print("No valid data points found for analysis")


In [ ]:
# import os
# import pickle

# # Create directory if it doesn't exist
# save_dir = "/app/data_saved/exp_buy_85_sell_89"
# os.makedirs(save_dir, exist_ok=True)

# # Save merged, m_dict, and b_dict
# with open(os.path.join(save_dir, "merged_buy.pkl"), "wb") as f:
#     pickle.dump(merged, f)

# with open(os.path.join(save_dir, "m_dict_buy.pkl"), "wb") as f:
#     pickle.dump(m_dict, f)

# with open(os.path.join(save_dir, "b_dict_buy.pkl"), "wb") as f:
#     pickle.dump(b_dict, f)

# print(f"Successfully saved merged, m_dict, and b_dict to {save_dir}")
# print(f"Files saved:")
# print(f"  - merged_buy.pkl")
# print(f"  - m_dict_buy.pkl") 
# print(f"  - b_dict_buy.pkl")


# Super debug

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# ================= CONFIG =================
DEBUG = True
MAX_IDS_TO_PRINT = 30
START_PREFIX_AT = 1
PREFIX_MODE = "le"         # "lt" -> insertion < a, "le" -> insertion <= a

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 50)

# ================ HELPERS ================
def _safe_col(points_df, preferred, fallbacks):
    if preferred in points_df.columns:
        return preferred
    for c in fallbacks:
        if c in points_df.columns:
            return c
    return None

def _safe_num(v, n=6):
    try:
        return float(np.round(v, n))
    except Exception:
        return v

def _print_section(h):
    print("\n" + "="*24 + f" {h} " + "="*24)

def _format_counts(mapping):
    items = [f"{{{int(k)}: {int(mapping[k])}}}" for k in sorted(mapping.keys())]
    return ", ".join(items) + "."

def _format_pairs_inline(series_like):
    pairs = [f"{int(k)}:{int(series_like.loc[k])}" for k in sorted(series_like.index)]
    return ", ".join(pairs) + "."

def debug_points_df(points_df, tag=""):
    _print_section(f"[DEBUG] points_df summary {tag}")
    if points_df is None:
        print("points_df is None")
        return
    print(f"shape: {points_df.shape}")
    cols = list(points_df.columns)
    print(f"columns[{len(cols)}]: {cols}")
    if len(points_df) == 0:
        print("points_df is EMPTY.")
        return
    with pd.option_context('display.max_columns', None, 'display.width', 200):
        print("head(5):")
        print(points_df.head(5))
        print("tail(5):")
        print(points_df.tail(5))
    col_i = _safe_col(points_df, "insertion", ["ins_idx", "insertion_idx", "idx"])
    if col_i:
        ii = pd.to_numeric(points_df[col_i], errors="coerce").dropna().astype(int)
        if len(ii):
            print(f"insertion: min={ii.min()}, max={ii.max()}, unique_count={ii.nunique()}, total={len(ii)}")
            cnt_by_ins = ii.value_counts().sort_index()
            print("insertion counts: " + _format_pairs_inline(cnt_by_ins))
    if "sample_id" in points_df.columns:
        sid = points_df["sample_id"].dropna()
        print(f"sample_id: unique={sid.nunique()}, total_rows_with_id={sid.shape[0]}")

def identity_checks(cnt_by_ins, counts_tail, counts_pref, ins_min, ins_max, total_points):
    _print_section("IDENTITY CHECKS")
    print("[A-DEBUG] points used per a for [a:]:")
    print(_format_counts(counts_tail))
    print("[A-DEBUG] points used per a for [:a]:")
    print(_format_counts(counts_pref))
    if PREFIX_MODE == "le":
        mismatches = []
        for a in range(ins_min, ins_max + 1):
            t_next = counts_tail.get(a + 1, 0) if (a + 1) <= ins_max else 0
            p_inc = counts_pref.get(a, 0)
            if t_next + p_inc != total_points:
                mismatches.append((a, t_next, p_inc))
        if mismatches:
            print("[CHECK] MISMATCHES:", mismatches)
        p_max = counts_pref.get(ins_max, 0)
        print(f"[DEMO] [:max] == total?  prefix[{ins_max}]={p_max}, total={total_points}, equal={p_max == total_points}")

# ================== CORE: replicate dashboard compute logic (no plotting) ==================
def _compute_points_and_coeffs_like_dashboard(
    b_dict_local, m_dict_local, *, hist_steps, gen_block, num_insertions, tick_size=100
):
    """
    Reproduces the x/y and per-sample alpha/beta from market_impact_dashboard_from_raw,
    including use of sample_day_map (H, L, execution_sum) and fixed-intercept per-sample beta.
    Returns points_df (sample_id, insertion, x, y) and coeffs_df (alpha_hat, beta_hat).
    """
    EVENT_TYPE_COL = 1
    PRICE_COL      = 3
    SIZE_COL       = 5
    eps = 1e-12
    tol = 1e-12

    sample_ids = sorted(set(b_dict_local.keys()) & set(m_dict_local.keys()))
    rows_points = []
    coeff_rows = []

    for sid in sample_ids:
        messages_ticks = m_dict_local[sid]
        book = b_dict_local[sid]  # not used, but kept for parity
        T = len(messages_ticks)

        # insertion schedule (same)
        insertion_positions = hist_steps + np.arange(1, num_insertions + 1) * gen_block
        valid_insertions = [pos for pos in insertion_positions if pos < T]
        if not valid_insertions:
            coeff_rows.append({"sample_id": sid, "alpha_hat": np.nan, "beta_hat": np.nan, "n_used": 0, "n_total": 0})
            continue

        # Reference price at first insertion (ticks -> $)
        ref_idx = valid_insertions[0]
        reference_price = float(messages_ticks[ref_idx, PRICE_COL]) / tick_size

        # ---- NEW: read H, L, execution_sum from sample_day_map (fallbacks identical to dashboard) ----
        try:
            day_row = sample_day_map[sample_day_map['sample_id'] == sid]
            if not day_row.empty:
                H_ticks = float(day_row.iloc[0]['highest_price'])
                L_ticks = float(day_row.iloc[0]['lowest_price'])
                execution_sum = float(day_row.iloc[0]['execution_sum'])
            else:
                H_ticks = float(np.max(messages_ticks[:hist_steps, PRICE_COL])) if hist_steps <= T else float(np.max(messages_ticks[:, PRICE_COL]))
                L_ticks = float(np.min(messages_ticks[:hist_steps, PRICE_COL])) if hist_steps <= T else float(np.min(messages_ticks[:, PRICE_COL]))
                exec_mask = (messages_ticks[:, EVENT_TYPE_COL].astype(int) == 4)
                execution_sum = float(np.sum(messages_ticks[exec_mask, SIZE_COL].astype(float)))
        except Exception:
            H_ticks = float(np.max(messages_ticks[:hist_steps, PRICE_COL])) if hist_steps <= T else float(np.max(messages_ticks[:, PRICE_COL]))
            L_ticks = float(np.min(messages_ticks[:hist_steps, PRICE_COL])) if hist_steps <= T else float(np.min(messages_ticks[:, PRICE_COL]))
            exec_mask = (messages_ticks[:, EVENT_TYPE_COL].astype(int) == 4)
            execution_sum = float(np.sum(messages_ticks[exec_mask, SIZE_COL].astype(float)))

        # Convert to dollars for Parkinson eta
        H = float(H_ticks) / tick_size
        L = float(L_ticks) / tick_size
        if np.isfinite(H) and np.isfinite(L) and H > L and L > 0:
            eta_day = np.log(H / L) / 0.8325546
            alpha_fixed = float(np.log(max(eta_day, eps)))   # α = ln(η)
        else:
            eta_day = eps
            alpha_fixed = float(np.log(eta_day))

        # Convert messages to dollars for helper functions
        messages_dollars = messages_ticks.astype(float).copy()
        messages_dollars[:, PRICE_COL] /= tick_size

        # ---- Use your helper functions exactly like in dashboard ----
        impact, vwap_series, Q_cum, log_imp = calculate_impact(
            messages_dollars, valid_insertions, reference_price
        )
        V_exp, log_qv = calculate_market_volume(
            messages_dollars, hist_steps, valid_insertions, execution_sum
        )

        # Collect per-insertion points
        mask_zero = impact <= tol
        mask_pos  = ~mask_zero
        used_x = []
        used_y = []
        for j, _idx in enumerate(valid_insertions):
            if mask_zero[j] or (not np.isfinite(log_qv[j])) or (not np.isfinite(log_imp[j])):
                continue
            rows_points.append({
                "sample_id": sid,
                "insertion": int(j + 1),
                "x": float(log_qv[j]),   # log(Q / V_exp) with V_exp from sample_day_map
                "y": float(log_imp[j])   # log(Impact)
            })
            used_x.append(log_qv[j])
            used_y.append(log_imp[j])

        used_x = np.asarray(used_x, dtype=float)
        used_y = np.asarray(used_y, dtype=float)
        n_used = int(np.sum(np.isfinite(used_x) & np.isfinite(used_y) & (used_x != 0)))
        n_total = int(len(valid_insertions))

        # per-sample β with fixed intercept using beta_fit function
        if n_used >= 2:
            valid_mask = np.isfinite(used_x) & np.isfinite(used_y) & (used_x != 0)
            if np.sum(valid_mask) >= 2:
                beta_hat = beta_fit(used_x[valid_mask], used_y[valid_mask], alpha_fixed, method=est_method)
            else:
                beta_hat = np.nan
        else:
            beta_hat = np.nan

        coeff_rows.append({
            "sample_id": sid,
            "alpha_hat": alpha_fixed,   # fixed ln(η_day)
            "beta_hat": beta_hat,
            "n_used": n_used,
            "n_total": n_total,
        })

    points_df = pd.DataFrame(rows_points)
    coeffs_df = pd.DataFrame.from_records(coeff_rows).set_index("sample_id").sort_index()
    return points_df, coeffs_df

# ================= MAIN (now uses new x/y and fixed-intercept slope) =================
cutoffs = [0.0, 0.2, 0.4, 0.6, 0.8]
filtered_sample_numbers = {}
fig = go.Figure()

for vcut in cutoffs:
    # Your existing filtering util (unchanged)
    x_filt, all_series_filt, merged_filt, hist_steps_filt, gen_block_filt = prepare_volatility_filtered_series(
        merged, hist_msgs, n_gen_msgs, midprice_step_size, volatility_cutoff=vcut
    )
    sample_ids = list(merged_filt['id'])
    filtered_sample_numbers[vcut] = sample_ids
    print(f"Cutoff {vcut}: {len(sample_ids)} samples")
    if DEBUG:
        ids_preview = sample_ids[:MAX_IDS_TO_PRINT]
        print(f"Sample IDs: {ids_preview}{' ...' if len(sample_ids) > MAX_IDS_TO_PRINT else ''}")

    # Filter your dicts
    b_dict_filt = {k: b_dict[k] for k in sample_ids if k in b_dict}
    m_dict_filt = {k: m_dict[k] for k in sample_ids if k in m_dict}

    # === NEW: compute points & coeffs with the exact same logic as the dashboard ===
    points_df, coeffs_df = _compute_points_and_coeffs_like_dashboard(
        b_dict_filt, m_dict_filt,
        hist_steps=hist_steps, gen_block=gen_block, num_insertions=num_insertions, tick_size=100
    )

    if DEBUG:
        debug_points_df(points_df, tag=f"(cutoff={vcut})")
    if points_df is None or len(points_df) == 0:
        continue

    # Column mapping
    COL_INS = "insertion" if "insertion" in points_df.columns else _safe_col(points_df, "insertion", ["ins_idx", "idx"])
    COL_X = "x" if "x" in points_df.columns else _safe_col(points_df, "x", ["log_qv"])
    COL_Y = "y" if "y" in points_df.columns else _safe_col(points_df, "y", ["log_impact"])

    ins_series = pd.to_numeric(points_df[COL_INS], errors="coerce").dropna().astype(int)
    ins_min, ins_max = int(ins_series.min()), int(ins_series.max())
    cnt_by_ins = ins_series.value_counts().sort_index()
    total_points = int(ins_series.shape[0])

    # === FIXED-INTERCEPT FIT: use α_global = mean(alpha_hat) exactly like the dashboard ===
    alpha_global = float(coeffs_df["alpha_hat"].mean(skipna=True)) if not coeffs_df.empty else 0.0

    def _fixed_intercept_beta(sub_df):
        """
        Compute beta with fixed intercept using beta_fit function
        (matching your dashboard's fit_for_mask logic)
        """
        if sub_df is None or len(sub_df) < 2:
            return np.nan
        X = pd.to_numeric(sub_df[COL_X], errors="coerce").to_numpy(dtype=float)
        Y = pd.to_numeric(sub_df[COL_Y], errors="coerce").to_numpy(dtype=float)
        valid = np.isfinite(X) & np.isfinite(Y) & (X != 0)
        if np.sum(valid) < 2:
            return np.nan
        return beta_fit(X[valid], Y[valid], alpha_global, method=est_method)

    # Tail [a:]
    a_tail = list(range(ins_min, ins_max + 1))
    betas_tail, counts_tail = [], {}
    for a in a_tail:
        sub = points_df[points_df[COL_INS] >= a]
        counts_tail[a] = len(sub)
        betas_tail.append(_fixed_intercept_beta(sub))

    # Prefix [:a] (<= or < depending on PREFIX_MODE)
    a_pref = list(range(max(START_PREFIX_AT, 1), ins_max + 1))
    betas_pref, counts_pref = [], {}
    for a in a_pref:
        sub = points_df[points_df[COL_INS] <= a] if PREFIX_MODE == "le" else points_df[points_df[COL_INS] < a]
        counts_pref[a] = len(sub)
        betas_pref.append(_fixed_intercept_beta(sub))

    # Exact-at-a
    a_exact = list(range(ins_min, ins_max + 1))
    betas_exact, counts_exact = [], {}
    for a in a_exact:
        sub = points_df[points_df[COL_INS] == a]
        counts_exact[a] = len(sub)
        betas_exact.append(_fixed_intercept_beta(sub))

    identity_checks(cnt_by_ins, counts_tail, counts_pref, ins_min, ins_max, total_points)

    # Plot with legend labels including cutoff
    fig.add_trace(go.Scatter(
        x=a_tail, y=betas_tail, mode="lines+markers",
        name=f"Vol:{vcut} [a:] (tail)"
    ))
    fig.add_trace(go.Scatter(
        x=a_pref, y=betas_pref, mode="lines+markers",
        name=f"Vol:{vcut} [:a] (prefix)"
    ))
    fig.add_trace(go.Scatter(
        x=a_exact, y=betas_exact, mode="lines+markers",
        name=f"Vol:{vcut} [exact @ a]"
    ))

# Guides
if len(fig.data) > 0:
    all_x = np.concatenate([np.asarray(tr.x, dtype=float) for tr in fig.data if len(tr.x)])
    x_min, x_max = int(np.nanmin(all_x)), int(np.nanmax(all_x))
else:
    x_min, x_max = 1, 20
fig.add_trace(go.Scatter(x=[x_min, x_max], y=[0.5, 0.5], mode="lines", line=dict(dash="dash"), name="y = 0.5"))
fig.add_trace(go.Scatter(x=[x_min, x_max], y=[0, 0], mode="lines", name="y = 0"))

fig.update_xaxes(title_text="a (insertion index threshold)", dtick=1)
fig.update_yaxes(title_text="β (slope)")
fig.update_layout(
    title=f"Global β vs a — tail, prefix, exact (PREFIX_MODE='{PREFIX_MODE}', method='{est_method}')",
    template="plotly_white", width=980, height=560
)
fig.show()

In [ ]:
# ================= SAMPLE SIZE ANALYSIS (tail [a:] only) =================
sample_sizes = [100, 200, 300, 400, "all"]
fig2 = go.Figure()

# Get all sample IDs (no volatility filtering)
all_sample_ids = list(merged['id'])
print(f"Total available samples: {len(all_sample_ids)}")

for sample_limit in sample_sizes:
    if sample_limit == "all":
        current_sample_ids = all_sample_ids
        label_suffix = f"n={len(current_sample_ids)}"
    else:
        current_sample_ids = all_sample_ids[:sample_limit]
        label_suffix = f"n={sample_limit}"
    
    print(f"Processing {label_suffix}")
    
    # Filter your dicts
    b_dict_filt = {k: b_dict[k] for k in current_sample_ids if k in b_dict}
    m_dict_filt = {k: m_dict[k] for k in current_sample_ids if k in m_dict}
    
    # Compute points & coeffs
    points_df, coeffs_df = _compute_points_and_coeffs_like_dashboard(
        b_dict_filt, m_dict_filt,
        hist_steps=hist_steps, gen_block=gen_block, num_insertions=num_insertions, tick_size=100
    )
    
    if DEBUG:
        debug_points_df(points_df, tag=f"({label_suffix})")
    if points_df is None or len(points_df) == 0:
        continue
    
    # Column mapping
    COL_INS = "insertion" if "insertion" in points_df.columns else _safe_col(points_df, "insertion", ["ins_idx", "idx"])
    COL_X = "x" if "x" in points_df.columns else _safe_col(points_df, "x", ["log_qv"])
    COL_Y = "y" if "y" in points_df.columns else _safe_col(points_df, "y", ["log_impact"])
    
    ins_series = pd.to_numeric(points_df[COL_INS], errors="coerce").dropna().astype(int)
    ins_min, ins_max = int(ins_series.min()), int(ins_series.max())
    
    # Global alpha for fixed-intercept fit
    alpha_global = float(coeffs_df["alpha_hat"].mean(skipna=True)) if not coeffs_df.empty else 0.0
    
    def _fixed_intercept_beta(sub_df):
        """
        Compute beta with fixed intercept using beta_fit function
        """
        if sub_df is None or len(sub_df) < 2:
            return np.nan
        X = pd.to_numeric(sub_df[COL_X], errors="coerce").to_numpy(dtype=float)
        Y = pd.to_numeric(sub_df[COL_Y], errors="coerce").to_numpy(dtype=float)
        valid = np.isfinite(X) & np.isfinite(Y) & (X != 0)
        if np.sum(valid) < 2:
            return np.nan
        beta = beta_fit(X[valid], Y[valid], alpha_global, method=est_method)
        return beta
    
    # Tail [a:] only
    a_tail = list(range(ins_min, ins_max + 1))
    betas_tail = []
    for a in a_tail:
        sub = points_df[points_df[COL_INS] >= a]
        betas_tail.append(_fixed_intercept_beta(sub))
    
    # Plot
    fig2.add_trace(go.Scatter(
        x=a_tail, y=betas_tail, mode="lines+markers",
        name=f"Samples {label_suffix}"
    ))

# Guides for second plot
if len(fig2.data) > 0:
    all_x = np.concatenate([np.asarray(tr.x, dtype=float) for tr in fig2.data if len(tr.x)])
    x_min, x_max = int(np.nanmin(all_x)), int(np.nanmax(all_x))
else:
    x_min, x_max = 1, 20
fig2.add_trace(go.Scatter(x=[x_min, x_max], y=[0.5, 0.5], mode="lines", line=dict(dash="dash"), name="y = 0.5"))
fig2.add_trace(go.Scatter(x=[x_min, x_max], y=[0, 0], mode="lines", name="y = 0"))

fig2.update_xaxes(title_text="a (insertion index threshold)", dtick=1)
fig2.update_yaxes(title_text="β (slope)")
fig2.update_layout(
    title="Global β vs a — Sample Size Analysis [a:] (tail)",
    template="plotly_white", width=980, height=560
)
fig2.show()
